In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import scipy.io as sio
import re
import torch
from tqdm.notebook import tqdm
from torch.utils.tensorboard import SummaryWriter

DATASET_PATH_STFT = "/content/drive/MyDrive/Datasets/SEED/de_stft.npz"
DATASET_PATH_BANDPASS = "/content/drive/MyDrive/Datasets/SEED/de_bandpass.npz"
DATASET_PATH_STFT_SMOOTH = "/content/drive/MyDrive/Datasets/SEED/de_stft_smooth.npz"
DATASET_PATH_BANDPASS_SMOOTH = "/content/drive/MyDrive/Datasets/SEED/de_bandpass_smooth.npz"

In [ ]:
%load_ext tensorboard

In [ ]:
from torch.utils.data import Dataset, DataLoader, Subset

class SEEDDataset(Dataset):
    def __init__(
        self,
        npz_path: str,
        person_ids: list[int] | None = None
    ):
        data = np.load(npz_path)
        X = data['X']
        y = data['y']
        persons = data['persons']
        sessions = data['sessions']

        if person_ids is not None:
            mask = np.isin(persons, person_ids)
            X = X[mask]
            y = y[mask]
            persons = persons[mask]
            sessions = sessions[mask]

        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
        self.y = torch.tensor(y, dtype=torch.long)
        self.persons = torch.tensor(persons, dtype=torch.long)
        self.sessions = torch.tensor(sessions, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self,idx: int):
        x = self.X[idx]
        y = self.y[idx]
        return x, y, self.persons[idx], self.sessions[idx]

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

class EEGResNet(nn.Module):
    def __init__(self, num_classes: int = 4):
        super(EEGResNet, self).__init__()

        self.backbone = models.resnet18(weights=None)

        self.backbone.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=64,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )

        self.backbone.maxpool = nn.Identity()
        self.backbone.fc = nn.Linear(self.backbone.fc.in_features, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone(x)


# Configuration

In [ ]:
model_config = {
    "model_name": "EEGResNet",
    "num_classes": 4,
    "batch_size": 32,
    "num_epochs": 50,
    "learning_rate": 1e-4,
    "optimizer_betas": (0.9, 0.999),
    "weight_decay": 1e-2,
    "dataset_type": "stft_smooth",
    "dataset": DATASET_PATH_STFT_SMOOTH,
}

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

class Trainer:
    def __init__(self,
        model: nn.Module,
        train_loader: torch.utils.data.DataLoader,
        test_loader: torch.utils.data.DataLoader,
        optimizer: torch.optim.Optimizer,
        criterion: nn.Module,
        device: torch.device,
        model_config: dict,
        best_model_path: str,
        log_dir: str) -> None:

        self.model = model
        self.train_loader = train_loader
        self.test_loader = test_loader
        self.optimizer = optimizer
        self.criterion = criterion
        self.device = device
        self.model_config = model_config
        self.best_model_path = best_model_path
        self.best_val_acc = 0.0

        # Initialize TensorBoard writer
        log_subdir = f"{self.model_config['model_name'].replace(' ', '_')}_{self.model_config['dataset_type']}_lr{str(self.model_config['learning_rate']).replace('.', '')}"
        self.writer = SummaryWriter(log_dir=os.path.join(log_dir, log_subdir))

    def train_epoch(self, epoch: int):
        self.model.train()
        train_loss, correct, total = 0.0, 0, 0

        loop = tqdm(self.train_loader, desc=f"Training Epoch {epoch}")

        for batch_idx, (x, y, _, _) in enumerate(loop):
            x, y = x.to(self.device), y.to(self.device)

            self.optimizer.zero_grad()
            out = self.model(x)
            loss = self.criterion(out, y)
            loss.backward()
            self.optimizer.step()

            train_loss += loss.item() * x.size(0)
            _, pred = out.max(1)
            total += y.size(0)
            correct += pred.eq(y).sum().item()

            loop.set_postfix(
                loss=train_loss / total,
                acc=100 * correct / total
            )

        train_acc = 100 * correct / total
        train_loss /= total
        return train_loss, train_acc

    def validate_epoch(self, epoch: int):
        self.model.eval()
        val_loss, cor, tot = 0.0, 0, 0

        with torch.no_grad():
            for x, y, _, _ in self.test_loader:
                x, y = x.to(self.device), y.to(self.device)
                out = self.model(x)
                loss = self.criterion(out, y)

                val_loss += loss.item() * x.size(0)
                _, pred = out.max(1)
                tot += y.size(0)
                cor += pred.eq(y).sum().item()

        val_acc = 100 * cor / tot
        val_loss /= tot
        return val_loss, val_acc

    def run(self):
        num_epochs = self.model_config["num_epochs"]

        for epoch in range(1, num_epochs + 1):
            print(f"\n=== Epoch {epoch}/{num_epochs} ===")

            train_loss, train_acc = self.train_epoch(epoch)
            val_loss, val_acc = self.validate_epoch(epoch)

            # Log epoch metrics to TensorBoard
            self.writer.add_scalar('Train/Loss', train_loss, epoch)
            self.writer.add_scalar('Train/Accuracy', train_acc, epoch)
            self.writer.add_scalar('Validation/Loss', val_loss, epoch)
            self.writer.add_scalar('Validation/Accuracy', val_acc, epoch)

            print(f"Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

            # Save best model
            if val_acc > self.best_val_acc:
                self.best_val_acc = val_acc
                torch.save(
                    {
                        "epoch": epoch,
                        "model_state": self.model.state_dict(),
                        "optimizer_state": self.optimizer.state_dict(),
                        "val_acc": val_acc
                    },
                    self.best_model_path
                )
                print(f"Best model saved (Val Acc = {val_acc:.2f}%)")
        self.writer.close()


# Partial Training

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset

TEST_SUBJECT_ID = 0
TRAIN_PERCENTAGE = 0.05

full_dataset = SEEDDataset(model_config["dataset"])

persons = full_dataset.persons.numpy()
labels = full_dataset.y.numpy()

subject_idx = np.where(persons == TEST_SUBJECT_ID)[0]
classes = np.unique(labels[subject_idx])

samples_per_class = int(len(subject_idx) * TRAIN_PERCENTAGE / len(classes))

rng = np.random.default_rng(42)

partial_train_idx = np.concatenate([
    rng.choice(
        subject_idx[labels[subject_idx] == class_id],
        size=samples_per_class,
        replace=False
    )
    for class_id in classes
])

partial_test_idx = np.setdiff1d(subject_idx, partial_train_idx)

other_idx = np.where(persons != TEST_SUBJECT_ID)[0]

train_indices = np.concatenate([
    other_idx,
    partial_train_idx
])

test_indices = partial_test_idx

train_dataset = Subset(full_dataset, train_indices)
test_dataset = Subset(full_dataset, test_indices)

train_loader = DataLoader(
    train_dataset,
    batch_size=model_config["batch_size"],
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=model_config["batch_size"],
    shuffle=False
)

print(f"Test subject: {TEST_SUBJECT_ID}")
print(f"Training percentage from subject: {TRAIN_PERCENTAGE * 100:.0f}%")
print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

num_epochs = model_config["num_epochs"]
device = "cuda" if torch.cuda.is_available() else "cpu"
num_classes = model_config["num_classes"]

model = EEGResNet(num_classes=num_classes)
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=model_config["learning_rate"],
    betas=model_config["optimizer_betas"],
    weight_decay=model_config["weight_decay"]
)

model_specs = f"{model_config['model_name'].replace(' ', '_')}_{model_config['dataset_type']}_lr{str(model_config['learning_rate']).replace('.', '')}"
METRICS_SAVE_PATH = f"/content/drive/MyDrive/Datasets/SEED/Train/Partial/metrics_{model_specs}/runs/"
BEST_MODEL_PATH = f"/content/drive/MyDrive/Datasets/SEED/Train/Partial/best_model_{model_specs}.pt"

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    model_config=model_config,
    best_model_path=BEST_MODEL_PATH,
    log_dir=METRICS_SAVE_PATH
)

In [ ]:
trainer.run()
print("Training complete.")

## Launch TensorBoard


In [ ]:
%tensorboard --logdir /content/drive/MyDrive/Datasets/SEED/Train/Partial/runs